In [1]:
import cv2
import os

print("OpenCV version:", cv2.__version__)

OpenCV version: 4.10.0


1 — Capture and display live webcam video

In [4]:
# Task 1: Live webcam stream
cap = cv2.VideoCapture(0)  # 0 = default webcam

if not cap.isOpened():
    print("Error: Could not open webcam. Check camera permissions or try VideoCapture(1).")
else:
    print("Press 'q' in the video window to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Could not receive a frame. Stopping.")
            break

        cv2.imshow("Live Webcam", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

Press 'q' in the video window to quit.


Face detection setup — Load the Haar Cascade classifier

In [10]:
cascade_file = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_file)

if face_cascade.empty():
    raise RuntimeError("Could not load Haar Cascade classifier: " + cascade_file)

print("Face detector loaded successfully.")

Face detector loaded successfully.


3 — Detect faces in real time using the webcam

In [11]:
# Task 3: Real-time face detection
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam.")
else:
    print("Press 'q' in the face-detection window to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Could not receive a frame.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        cv2.putText(frame, f"Faces: {len(faces)}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow("Real-Time Face Detection", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

Press 'q' in the face-detection window to quit.


5 — Build a simple webcam motion detector

In [14]:
# 5: Webcam motion detection
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam.")
else:
    previous_gray = None
    min_area = 800  # Increase to ignore smaller movements
    print("Move in front of the camera. Press 'q' in the window to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Could not receive a frame.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)

        if previous_gray is not None:
            # Compare current frame with the previous frame
            frame_difference = cv2.absdiff(previous_gray, gray)
            _, threshold = cv2.threshold(frame_difference, 25, 255, cv2.THRESH_BINARY)
            threshold = cv2.dilate(threshold, None, iterations=2)

            contours, _ = cv2.findContours(
                threshold, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )

            motion_regions = 0
            for contour in contours:
                if cv2.contourArea(contour) < min_area:
                    continue

                x, y, w, h = cv2.boundingRect(contour)
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
                motion_regions += 1

            cv2.putText(frame, f"Motion regions: {motion_regions}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        previous_gray = gray
        cv2.imshow("Webcam Motion Detector", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

Move in front of the camera. Press 'q' in the window to quit.
